# Test WFS Data Bundler

**Goal**: Test the new `WFSDataBundler` class that abstracts WFS data collection across multiple regions.

**What this does**:
1. Fetches WFS data for multiple scope regions using `DataCollector` under the hood
2. Aggregates and deduplicates data across regions
3. Saves to GeoPackage with proper naming: `wfs_{service}_{layer}`
4. Provides checkpointing support for long runs

**Why use this**:
- Cleaner code - no notebook boilerplate
- Reusable - can be called from scripts or other notebooks
- Production-ready - includes error handling, logging, retry logic
- Scalable - designed for 12,130 regions

## Setup

In [18]:
import sys
sys.path.append("../")

import src.paths as PATHS
import src.data.wfs_bundler as WFS_BUNDLER
import src.data.config as DATA_CONFIG

import geopandas as gpd
from pathlib import Path
from datetime import datetime
import shutil
import logging

# Setup logging to see bundler output
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

print("✅ Setup complete!")

✅ Setup complete!


## 1. Load Scope Regions

We'll start with a small test to verify everything works.

In [19]:
# Choose your test dataset
# Option 1: Full phase 2 dataset (12,130 regions)
# filename = "wocu_output_fase2_v4"

# Option 2: Phase 1 dataset (smaller - 233 regions)
filename = "phase1_2025-08-14_v1"

# Option 3: Luke's test set (11 regions - fastest)
# filename = "luke_inputs_v3"

gpkg_path = PATHS.DATA_DIR / f"{filename}.gpkg"

# Create a copy with date suffix so we don't modify the original
date_suffix = datetime.now().strftime("%Y%m%d")
output_path = PATHS.DATA_DIR / f"{filename}_w_features_{date_suffix}.gpkg"

# Copy original file
if not output_path.exists():
    shutil.copy2(gpkg_path, output_path)
    print(f"📁 Created working copy: {output_path.name}")
else:
    print(f"📁 Using existing file: {output_path.name}")

# Load scope regions
scope_regions = gpd.read_file(output_path, layer="vlakken_scope")

print(f"\n📍 Loaded {len(scope_regions)} scope regions")
print(f"📐 CRS: {scope_regions.crs}")

# Find the ID column (different geopackages use different names)
id_column = None
for col in ['position_id', 'location_id', 'region_id', 'id']:
    if col in scope_regions.columns:
        id_column = col
        break

if id_column:
    print(f"\n🗺️ ID column: '{id_column}'")
    print(f"   First 3 IDs: {scope_regions[id_column].head(3).tolist()}")
else:
    print(f"\n🗺️ No standard ID column found - will use generated IDs (region_0, region_1, ...)")
    print(f"   Available columns: {list(scope_regions.columns)}")

scope_regions.head()

📁 Using existing file: phase1_2025-08-14_v1_w_features_20260127.gpkg

📍 Loaded 233 scope regions
📐 CRS: EPSG:28992

🗺️ ID column: 'location_id'
   First 3 IDs: ['maas_l_2180_2181', 'maas_l_2181_2182', 'maas_l_2182_2183']


,location_id,start_year,end_year,geometry
0,maas_l_2180_2181,2016,2024,"POLYGON ((148604.881 416309.328, 148538.207 41..."
1,maas_l_2181_2182,2016,2024,"POLYGON ((148491.678 416329.661, 148416.977 41..."
2,maas_l_2182_2183,2016,2024,"POLYGON ((148373.399 416364.687, 148290.662 41..."
3,maas_l_2183_2184,2016,2024,"POLYGON ((148268.941 416410.637, 148182.141 41..."
4,maas_l_2184_2185,2016,2024,"POLYGON ((148182.141 416457.702, 148095.34 416..."


## 2. Configure WFS Services

Using the default configuration which includes:
- Land Use (BRP Gewaspercelen)
- Buildings (BAG)
- Vegetation Legger (3 layers: bomen, heggen, vegetatieklassen)

In [10]:
# Load the default configuration
config = DATA_CONFIG.DataConfiguration()

print("📡 WFS Services configured:")
for i, wfs_service in enumerate(config.known_wfs_services, 1):
    print(f"\n{i}. {wfs_service.name}")
    print(f"   URL: {wfs_service.url}")
    print(f"   Layers: {', '.join(wfs_service.relevant_layers)}")

print(f"\n🔧 Buffer distance: {config.prediction_region_buffer}m")

📡 WFS Services configured:

1. land_use
   URL: https://service.pdok.nl/rvo/brpgewaspercelen/wfs/v1_0
   Layers: BrpGewas

2. building_location
   URL: https://service.pdok.nl/lv/bag/wfs/v2_0
   Layers: bag:pand

3. vegetation
   URL: https://geo.rijkswaterstaat.nl/services/ogc/gdr/rws_vegetatielegger/ows?version=2.0.0
   Layers: rws_vegetatielegger:bomen, rws_vegetatielegger:heggen, rws_vegetatielegger:vegetatieklassen

🔧 Buffer distance: 10m


## 3. Initialize WFS Data Bundler

In [11]:
# Create bundler instance
bundler = WFS_BUNDLER.WFSDataBundler(
    scope_regions=scope_regions,
    config=config,
    wfs_timeout=30,  # 30 seconds per WFS request
    max_retries=3,   # Retry failed requests 3 times
)

print("✅ WFSDataBundler initialized!")

INFO: Initialized WFSDataBundler with 11 regions


✅ WFSDataBundler initialized!


## 4. Fetch WFS Data

**TEST MODE**: Start by fetching just 10 regions to verify everything works.

Expected timing:
- 10 regions × 5 layers ≈ 30-60 seconds
- 100 regions × 5 layers ≈ 5-10 minutes
- 233 regions × 5 layers ≈ 10-20 minutes
- 12,130 regions × 5 layers ≈ 7-10 hours

In [12]:
# Fetch data - start with TEST MODE!
wfs_data, region_ids, successful, failed = bundler.fetch_all_regions(
    test_mode=True,        # Change to False for full run
    num_test=10,           # Number of regions to test
    show_progress=True,    # Show progress bar
)

print("\n" + "="*50)
print("✅ Fetch complete!")
print("="*50)

INFO: Processing 10 scope regions...
INFO: DataCollector: 30s timeout, 3 retries per service
Fetching WFS data:   0%|          | 0/10 [00:00<?, ?it/s]INFO: Getting data from the WFS service land_use.
INFO: Getting data from the layer BrpGewas in land_use
INFO: Getting features 0 to 5.
INFO: Getting data from the WFS service building_location.
INFO: Getting data from the layer bag:pand in building_location
INFO: Getting data from the WFS service vegetation.
INFO: Getting data from the layer rws_vegetatielegger:bomen in vegetation
INFO: Getting data from the layer rws_vegetatielegger:heggen in vegetation
INFO: Getting data from the layer rws_vegetatielegger:vegetatieklassen in vegetation
INFO: Getting features 0 to 11.
Fetching WFS data:  10%|█         | 1/10 [00:01<00:15,  1.69s/it]INFO: Getting data from the WFS service land_use.
INFO: Getting data from the layer BrpGewas in land_use
INFO: Getting features 0 to 3.
INFO: Getting data from the WFS service building_location.
INFO: Getting


✅ Fetch complete!


## 5. View Summary Statistics

In [13]:
# Get detailed statistics
stats = bundler.get_summary_stats()

print("📊 Summary Statistics:")
print(f"\n  Regions processed: {stats['num_regions_processed']}")
print(f"  Successful: {stats['num_successful']}")
print(f"  Failed: {stats['num_failed']}")
print(f"  Services: {stats['num_services']}")

print("\n📡 Data by Service:")
for service_name, service_stats in stats['services'].items():
    print(f"\n  {service_name}:")
    print(f"    Layers: {service_stats['num_layers']}")
    for layer_name, layer_stats in service_stats['layers'].items():
        print(f"      - {layer_name}: {layer_stats['total_features']} features from {layer_stats['num_regions']} regions")

📊 Summary Statistics:

  Regions processed: 10
  Successful: 10
  Failed: 0
  Services: 3

📡 Data by Service:

  land_use:
    Layers: 1
      - BrpGewas: 25 features from 10 regions

  building_location:
    Layers: 1
      - bag:pand: 0 features from 0 regions

  vegetation:
    Layers: 3
      - rws_vegetatielegger:bomen: 1 features from 1 regions
      - rws_vegetatielegger:heggen: 0 features from 0 regions
      - rws_vegetatielegger:vegetatieklassen: 88 features from 10 regions


## 6. Save to GeoPackage

This will:
- Deduplicate geometries that span multiple regions
- Add `scope_region_id` column to track which region fetched each feature
- Save with naming convention: `wfs_{service}_{layer}`

In [14]:
# Save to GeoPackage
saved_layers = bundler.save_to_geopackage(
    output_path=output_path,
    add_region_ids=True,  # Track which region each feature came from
)

print("\n" + "="*50)
print(f"✅ Saved {len(saved_layers)} layers!")
print("="*50)

INFO: Saving WFS data to luke_inputs_v3_w_features_20260127.gpkg...
INFO: Processing service: land_use
INFO: Created 7 records
INFO: Saved 7 features to layer: wfs_land_use_BrpGewas (removed 18 duplicates, 72.0%)
INFO: Processing service: building_location
INFO: Processing service: vegetation
INFO: Created 1 records
INFO: Saved 1 features to layer: wfs_vegetation_rws_vegetatielegger_bomen
INFO: Created 22 records
INFO: Saved 22 features to layer: wfs_vegetation_rws_vegetatielegger_vegetatieklassen (removed 66 duplicates, 75.0%)
INFO: DONE! Saved 3 new layers to luke_inputs_v3_w_features_20260127.gpkg
INFO: New layers:
INFO:   - wfs_land_use_BrpGewas
INFO:   - wfs_vegetation_rws_vegetatielegger_bomen
INFO:   - wfs_vegetation_rws_vegetatielegger_vegetatieklassen



✅ Saved 3 layers!


## 7. Verify Saved Layers

In [15]:
# List all layers in the GeoPackage
layers = bundler.list_geopackage_layers(output_path)

print(f"📦 Layers in {output_path.name}:\n")

print(f"📂 Original layers ({len(layers['original'])})")
for layer in layers['original']:
    print(f"  - {layer}")

print(f"\n🆕 WFS layers ({len(layers['wfs'])})")
for layer in layers['wfs']:
    print(f"  - {layer}")

📦 Layers in luke_inputs_v3_w_features_20260127.gpkg:

📂 Original layers (4)
  - punten_oever
  - samenvatting
  - vlakken_erosie
  - vlakken_scope

🆕 WFS layers (14)
  - wfs_land_use_BrpGewas
  - wfs_rws_legger_rws_legger_begrenzing_rijksvaarweg_legger
  - wfs_rws_legger_rws_legger_bladkader_legger
  - wfs_rws_legger_rws_legger_dwarsprofiel_over_genormeerde_bodem_legger
  - wfs_rws_legger_rws_legger_genormeerd_bodem_legger
  - wfs_rws_legger_rws_legger_krib_legger
  - wfs_rws_legger_rws_legger_kribhoogtes_legger
  - wfs_rws_legger_rws_legger_nevengeul_strang_legger
  - wfs_rws_legger_rws_legger_rws_vrijeruimtenevgeulstrang_v
  - wfs_rws_legger_rws_legger_tussengrens_oppervlaktewaterlichamen_legger
  - wfs_rws_legger_rws_legger_water_legger
  - wfs_rws_legger_rws_legger_zomer_kade_legger
  - wfs_vegetation_rws_vegetatielegger_bomen
  - wfs_vegetation_rws_vegetatielegger_vegetatieklassen


## 8. Inspect a Sample Layer

In [16]:
# Pick the first WFS layer to inspect
if saved_layers:
    sample_layer = saved_layers[0]
    print(f"🔍 Inspecting layer: {sample_layer}\n")
    
    sample_gdf = gpd.read_file(output_path, layer=sample_layer)
    
    print(f"📊 Shape: {sample_gdf.shape}")
    print(f"📐 CRS: {sample_gdf.crs}")
    print(f"\n🗂️ Columns: {list(sample_gdf.columns)}")
    print(f"\n📍 First 5 rows:")
    display(sample_gdf.head())
else:
    print("⚠️ No layers were saved")

🔍 Inspecting layer: wfs_land_use_BrpGewas

📊 Shape: (7, 7)
📐 CRS: EPSG:28992

🗂️ Columns: ['category', 'gewas', 'gewascode', 'jaar', 'status', 'scope_region_id', 'geometry']

📍 First 5 rows:


,category,gewas,gewascode,jaar,status,scope_region_id,geometry
0,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",331,2024,Definitief,waal_949_0_949_1_left,"POLYGON ((132479.628 424934.482, 132481.759 42..."
1,Natuurterrein,Natuurterreinen (incl. heide),335,2024,Definitief,waal_949_0_949_1_left,"POLYGON ((132308.214 425512.189, 132310.468 42..."
2,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",331,2024,Definitief,waal_949_0_949_1_left,"POLYGON ((132223.63 425416.776, 132421.559 425..."
3,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",331,2024,Definitief,waal_949_0_949_1_left,"POLYGON ((131310 425022.76, 131319.695 425025...."
4,Grasland,"Grasland, natuurlijk. Met landbouwactiviteiten.",331,2024,Definitief,waal_949_0_949_1_left,"POLYGON ((131498.32 424825.807, 131499.352 424..."


---

## ✅ Next Steps

### If test run succeeded:

1. **Open in QGIS**: Load the GeoPackage and visually inspect the `wfs_*` layers
2. **Verify data quality**: 
   - Do geometries look correct?
   - Are they in the right locations?
   - Compare with scope regions layer
3. **Scale up gradually**:
   - Try 100 regions (set `num_test=100`)
   - Try 500 regions to estimate full run time
   - Run full dataset overnight (set `test_mode=False`)

### For production runs (12,130 regions):

1. **Run overnight** - estimated 7-10 hours
2. **Monitor progress** - check logs for failures
3. **Validate in QGIS** before using in DataHandler
4. **Then implement** `DataHandler.load_remote_data_from_geopackage()`

### Expected output layers:
- `wfs_land_use_BrpGewas` - Agricultural crop parcels
- `wfs_building_location_bag_pand` - Building footprints
- `wfs_vegetation_rws_vegetatielegger_bomen` - Trees
- `wfs_vegetation_rws_vegetatielegger_heggen` - Hedges
- `wfs_vegetation_rws_vegetatielegger_vegetatieklassen` - Vegetation classes

---

## Optional: Full Run (No Test Mode)

⚠️ **ONLY RUN THIS AFTER VERIFYING TEST MODE WORKS**

Uncomment and run the cell below for a full production run.

In [17]:
# Full production run - NO TEST MODE
bundler_full = WFS_BUNDLER.WFSDataBundler(
    scope_regions=scope_regions,
    config=config,
    wfs_timeout=30,
    max_retries=3,
)

# Fetch all regions
wfs_data, region_ids, successful, failed = bundler_full.fetch_all_regions(
    test_mode=False,  # Full run!
    show_progress=True,
)

# Save results
saved_layers = bundler_full.save_to_geopackage(
    output_path=output_path,
    add_region_ids=True,
)

print(f"\n✅ DONE! Saved {len(saved_layers)} layers for {len(scope_regions)} regions")

INFO: Initialized WFSDataBundler with 11 regions
INFO: Processing 11 scope regions...
INFO: DataCollector: 30s timeout, 3 retries per service
Fetching WFS data:   0%|          | 0/11 [00:00<?, ?it/s]INFO: Getting data from the WFS service land_use.
INFO: Getting data from the layer BrpGewas in land_use
INFO: Getting features 0 to 5.
INFO: Getting data from the WFS service building_location.
INFO: Getting data from the layer bag:pand in building_location
INFO: Getting data from the WFS service vegetation.
INFO: Getting data from the layer rws_vegetatielegger:bomen in vegetation
INFO: Getting data from the layer rws_vegetatielegger:heggen in vegetation
INFO: Getting data from the layer rws_vegetatielegger:vegetatieklassen in vegetation
INFO: Getting features 0 to 11.
Fetching WFS data:   9%|▉         | 1/11 [00:01<00:15,  1.51s/it]INFO: Getting data from the WFS service land_use.
INFO: Getting data from the layer BrpGewas in land_use
INFO: Getting features 0 to 3.
INFO: Getting data from


✅ DONE! Saved 3 layers for 11 regions
